In [0]:
from pyspark.sql import functions as F

path = "dbfs:/databricks-datasets/sms_spam_collection/data-001/smsData.csv"

# 1) Wczytanie i rozdzielenie na label + text
df = (spark.read.text(path)
    .withColumn("parts", F.split(F.col("value"), "\t", 2))
    .withColumn("label", F.col("parts").getItem(0))
    .withColumn("text", F.col("parts").getItem(1))
    .select("label", "text")
    .dropna()
)


# 2) lower + czyszczenie unicode
df = (df
    .withColumn("text_lc", F.lower(F.col("text")))
    .withColumn("text_norm", F.regexp_replace("text_lc", r"[^\x00-\x7F]+", " "))
)

# 3) Flagi cech
df = (df
    .withColumn("HAS_URL", F.col("text_norm").rlike(
        r"(?i)\b((https?://|www\.)\S+|[a-z0-9.-]+\.(com|org|net|edu|gov|co|uk|pl|io|me|info|biz|sg)\b)"
    ))
    .withColumn("HAS_NUMBER", F.col("text_norm").rlike(r"\b\d+([.,]\d+)?\b"))
    .withColumn("HAS_CURRENCY",F.col("text_lc").rlike(r"(\$|£|€|\bgbp\b|\beur\b|\busd\b|\bpound\b|\beuro\b)")
)
)

# 4) Czyszczenie + tokenizacja
df = (df
    .withColumn("clean", F.regexp_replace("text_norm", r"[^a-z0-9\s]+", " "))
    .withColumn("clean", F.regexp_replace("clean", r"\s+", " "))
    .withColumn("clean", F.trim("clean"))
    .withColumn("tokens", F.expr("filter(split(clean, ' '), x -> x <> '')"))
)

# 5) Budowa transakcji: tokens + flagi
df = df.withColumn("items", F.col("tokens"))

df = df.withColumn(
    "items",
    F.when(F.col("HAS_URL"), F.array_union(F.col("items"), F.array(F.lit("HAS_URL"))))
     .otherwise(F.col("items"))
)

df = df.withColumn(
    "items",
    F.when(F.col("HAS_NUMBER"), F.array_union(F.col("items"), F.array(F.lit("HAS_NUMBER"))))
     .otherwise(F.col("items"))
)

df = df.withColumn(
    "items",
    F.when(F.col("HAS_CURRENCY"), F.array_union(F.col("items"), F.array(F.lit("HAS_CURRENCY"))))
     .otherwise(F.col("items"))
)

df = (
    df
    # usuwamy duplikaty w transakcji
    .withColumn("items", F.array_distinct("items"))
    
    # usuwamy tokeny null
    .withColumn("items", F.expr("filter(items, x -> x is not null)"))
    
    # usuwamy bardzo krótkie tokeny (np. "u", "x", "2")
    .withColumn("items", F.expr("filter(items, x -> length(x) >= 3)"))
    
    # zostawiamy tylko sensowne transakcje
    .filter(F.size("items") >= 2)
)


df = df.select("label", "text", "items", "HAS_URL", "HAS_NUMBER", "HAS_CURRENCY")

display(df.limit(10))


label,text,items,HAS_URL,HAS_NUMBER,HAS_CURRENCY
ham,Ok... But they said i've got wisdom teeth hidden inside n mayb need 2 remove.,"List(but, they, said, got, wisdom, teeth, hidden, inside, mayb, need, remove, HAS_NUMBER)",false,true,false
ham,� thk of wat to eat tonight.,"List(thk, wat, eat, tonight)",false,false,false
ham,I dunno until when... Lets go learn pilates...,"List(dunno, until, when, lets, learn, pilates)",false,false,false
spam,Someonone you know is trying to contact you via our dating service! To find out who it could be call from your mobile or landline 09064015307 BOX334SK38ch,"List(someonone, you, know, trying, contact, via, our, dating, service, find, out, who, could, call, from, your, mobile, landline, 09064015307, box334sk38ch, HAS_NUMBER)",false,true,false
spam,URGENT! We are trying to contact U. Todays draw shows that you have won a �800 prize GUARANTEED. Call 09050003091 from land line. Claim C52. Valid12hrs only,"List(urgent, are, trying, contact, todays, draw, shows, that, you, have, won, 800, prize, guaranteed, call, 09050003091, from, land, line, claim, c52, valid12hrs, only, HAS_NUMBER)",false,true,false
spam,Not heard from U4 a while. Call 4 rude chat private line 01223585334 to cum. Wan 2C pics of me gettin shagged then text PIX to 8552. 2End send STOP 8552 SAM xxx,"List(not, heard, from, while, call, rude, chat, private, line, 01223585334, cum, wan, pics, gettin, shagged, then, text, pix, 8552, 2end, send, stop, sam, xxx, HAS_NUMBER)",false,true,false
ham,staff.science.nus.edu.sg/~phyhcmk/teaching/pc1323,"List(staff, science, nus, edu, phyhcmk, teaching, pc1323, HAS_URL)",true,false,false
ham,Thank god they are in bed!,"List(thank, god, they, are, bed)",false,false,false
ham,Hey tmr meet at bugis 930 ?,"List(hey, tmr, meet, bugis, 930, HAS_NUMBER)",false,true,false
spam,You are a winner you have been specially selected to receive �1000 cash or a �2000 award. Speak to a live operator to claim call 087123002209am-7pm. Cost 10p,"List(you, are, winner, have, been, specially, selected, receive, 1000, cash, 2000, award, speak, live, operator, claim, call, 087123002209am, 7pm, cost, 10p, HAS_NUMBER)",false,true,false


In [0]:
display(
    df.select(
        F.avg(F.col("HAS_URL").cast("double")).alias("share_HAS_URL"),
        F.avg(F.col("HAS_NUMBER").cast("double")).alias("share_HAS_NUMBER"),
        F.avg(F.col("HAS_CURRENCY").cast("double")).alias("share_HAS_CURRENCY")
    )
)

share_HAS_URL,share_HAS_NUMBER,share_HAS_CURRENCY
0.04302788844621514,0.502788844621514,0.010358565737051793


In [0]:
df = df.withColumn("items", F.expr("filter(items, x -> x <> 'HAS_NUMBER')")) # HAS NUMBER dominuje z supportem blisko 50%, wiec na rzecz czytelnych wynikow usuwam z modelu


minsup1 = 0.005
minsup2  = 0.01

N = df.count()
print("Liczba transakcji:", N)
print("min_count dla 0.005:", int(minsup1 * N))
print("min_count dla 0.01 :", int(minsup2  * N))

Liczba transakcji: 1255
min_count dla 0.005: 6
min_count dla 0.01 : 12


In [0]:
L1 = (df
    .select(F.explode("items").alias("item"))
    .groupBy("item")
    .agg(F.count("*").alias("count"))
    .withColumn("support", F.col("count")/F.lit(N))
)

L1_005 = L1.filter(F.col("support") >= F.lit(minsup1)).orderBy(F.desc("support"))
L1_01  = L1.filter(F.col("support") >= F.lit(minsup2)).orderBy(F.desc("support"))

print("L1 minsup=0.005:", L1_005.count())
print("L1 minsup=0.01 :", L1_01.count())

display(L1_005.limit(30))

L1 minsup=0.005: 437
L1 minsup=0.01 : 238


item,count,support
call,208,0.1657370517928287
now,179,0.14262948207171314
you,164,0.1306772908366534
lor,140,0.11155378486055777
the,136,0.10836653386454183
your,109,0.08685258964143426
but,106,0.08446215139442231
then,104,0.08286852589641434
can,103,0.08207171314741035
have,97,0.07729083665338646


In [0]:
N = df.count()

pairs = (
    df
    .withColumn("items_sorted", F.array_sort("items"))
    .select("*", F.posexplode("items_sorted").alias("i", "w1"))  # i = 0..len-1
    .withColumn("j", F.explode(F.sequence(F.col("i") + 1, F.size("items_sorted") - 1)))  # j = i+1..len-1
    .withColumn("w2", F.expr("get(items_sorted, j)"))  # 0-based indexing, safe
    .select(F.array("w1", "w2").alias("itemset"))
)

L2 = (pairs
    .groupBy("itemset")
    .agg(F.count("*").alias("count"))
    .withColumn("support", F.col("count") / F.lit(N))
)

L2_1 = L2.filter(F.col("support") >= F.lit(minsup1)).orderBy(F.desc("support"))
L2_2 = L2.filter(F.col("support") >= F.lit(minsup2)).orderBy(F.desc("support"))

print("L2 minsup1:", L2_1.count())
print("L2 minsup2:", L2_2.count())

display(L2_1.limit(30))

L2 minsup1: 1230
L2 minsup2: 279


itemset,count,support
"List(you, you)",118,0.09402390438247012
"List(you, null)",118,0.09402390438247012
"List(your, null)",107,0.08525896414342629
"List(your, your)",107,0.08525896414342629
"List(call, you)",97,0.07729083665338646
"List(call, your)",68,0.054183266932270914
"List(call, now)",56,0.04462151394422311
"List(call, have)",55,0.043824701195219126
"List(wat, null)",54,0.04302788844621514
"List(wat, wat)",54,0.04302788844621514


In [0]:

triples = (
    df
    .withColumn("items_sorted", F.array_sort("items"))
    .select("*", F.posexplode("items_sorted").alias("i", "w1"))
    .withColumn("j", F.explode(F.sequence(F.col("i") + 1, F.size("items_sorted") - 2)))
    .withColumn("w2", F.expr("get(items_sorted, j)"))
    .withColumn("k", F.explode(F.sequence(F.col("j") + 1, F.size("items_sorted") - 1)))
    .withColumn("w3", F.expr("get(items_sorted, k)"))
    .select(F.array("w1", "w2", "w3").alias("itemset"))
)

triples = triples.filter(F.expr("itemset[0] is not null and itemset[1] is not null and itemset[2] is not null"))
triples = triples.withColumn("itemset", F.array_distinct("itemset"))
triples = triples.filter(F.size("itemset") == 3)

L3 = (
    triples
    .groupBy("itemset")
    .agg(F.count("*").alias("count"))
    .withColumn("support", F.col("count") / F.lit(N))
)

L3_1 = L3.filter(F.col("support") >= F.lit(minsup1)).orderBy(F.desc("support"))
L3_2 = L3.filter(F.col("support") >= F.lit(minsup2)).orderBy(F.desc("support"))

print("L3 minsup1:", L3_1.count())
print("L3 minsup2:", L3_2.count())

display(L3_1.limit(30))


L3 minsup1: 1983
L3 minsup2: 161


itemset,count,support
"List(call, have, you)",38,0.030278884462151396
"List(are, call, you)",34,0.027091633466135457
"List(call, prize, won)",34,0.027091633466135457
"List(call, prize, you)",33,0.026294820717131476
"List(call, claim, have)",33,0.026294820717131476
"List(call, have, won)",31,0.024701195219123506
"List(call, have, prize)",30,0.02390438247011952
"List(call, claim, prize)",30,0.02390438247011952
"List(have, prize, won)",29,0.023107569721115537
"List(call, won, you)",29,0.023107569721115537


In [0]:
minconf1 = 0.6
minconf2 = 0.8

# support dla pojedynczych elementów
supp_ante = L1.select(F.col("item").alias("ante"), F.col("support").alias("supp_ante"))
supp_cons = L1.select(F.col("item").alias("cons"), F.col("support").alias("supp_cons"))

# Reguły z par: a=>b i b=>a, support(a,b) = support z L2
rules = (
    L2
    .withColumn("a", F.col("itemset").getItem(0))
    .withColumn("b", F.col("itemset").getItem(1))
    .select(F.col("a").alias("ante"), F.col("b").alias("cons"), F.col("support").alias("supp_ab"))
    .unionByName(
        L2
        .withColumn("a", F.col("itemset").getItem(0))
        .withColumn("b", F.col("itemset").getItem(1))
        .select(F.col("b").alias("ante"), F.col("a").alias("cons"), F.col("support").alias("supp_ab"))
    )
)

# confidence = supp_ab / supp_ante
# lift = confidence / supp_cons
rules = (
    rules
    .join(supp_ante, on="ante", how="left")
    .join(supp_cons, on="cons", how="left")
    .withColumn("confidence", F.col("supp_ab") / F.col("supp_ante"))
    .withColumn("lift", F.col("confidence") / F.col("supp_cons"))
)

# Filtr po minconf, supp_ab i ante != cons
rules_c1 = rules.filter(F.col("confidence") >= F.lit(minconf1)).filter(F.col("supp_ab") >= F.lit(minsup1)).filter(F.col("ante")!=F.col("cons")).orderBy(F.desc("lift"), F.desc("confidence"))
rules_c2 = rules.filter(F.col("confidence") >= F.lit(minconf2)).filter(F.col("supp_ab") >= F.lit(minsup1)).filter(F.col("ante")!=F.col("cons")).orderBy(F.desc("lift"), F.desc("confidence"))

print("Liczba reguł (minconf=0.6):", rules_c1.count())
print("Liczba reguł (minconf=0.8):", rules_c2.count())

display(rules_c1.select("ante","cons","supp_ab","confidence","lift").limit(10))
display(rules_c2.select("ante","cons","supp_ab","confidence","lift").limit(10))


Liczba reguł (minconf=0.6): 368
Liczba reguł (minconf=0.8): 160


ante,cons,supp_ab,confidence,lift
08000930705,camcorder,0.007171314741035857,1.0,139.44444444444446
camcorder,08000930705,0.007171314741035857,1.0,139.44444444444446
ldew,87066,0.005577689243027889,1.0,125.5
87066,ldew,0.005577689243027889,0.7000000000000001,125.5
2lands,w1j6hl,0.006374501992031873,0.7272727272727273,114.0909090909091
suite342,w1j6hl,0.006374501992031873,0.7272727272727273,114.0909090909091
2lands,suite342,0.008764940239043825,1.0,114.09090909090908
750,tried,0.005577689243027889,1.0,114.09090909090908
w1j6hl,suite342,0.006374501992031873,1.0,114.09090909090908
suite342,2lands,0.008764940239043825,1.0,114.09090909090908


ante,cons,supp_ab,confidence,lift
camcorder,08000930705,0.007171314741035857,1.0,139.44444444444446
08000930705,camcorder,0.007171314741035857,1.0,139.44444444444446
ldew,87066,0.005577689243027889,1.0,125.5
w1j6hl,2lands,0.006374501992031873,1.0,114.09090909090908
2lands,suite342,0.008764940239043825,1.0,114.09090909090908
suite342,2lands,0.008764940239043825,1.0,114.09090909090908
750,tried,0.005577689243027889,1.0,114.09090909090908
w1j6hl,suite342,0.006374501992031873,1.0,114.09090909090908
2lands,row,0.008764940239043825,1.0,96.53846153846153
await,sae,0.008764940239043825,1.0,96.53846153846153


In [0]:
words = (
    df
    .select("label", F.explode("items").alias("word"))
    .filter(F.col("word").isNotNull())
)

top_words = (
    words
    .groupBy("label", "word")
    .count()
    .orderBy(F.desc("count"))
)

display(top_words)

label,word,count
spam,call,167
ham,lor,140
spam,you,119
ham,but,105
ham,then,102
ham,can,94
spam,your,93
spam,now,90
ham,now,89
ham,got,83


In [0]:
spam_words = (
    top_words
    .filter(F.col("label") == "spam")
    .select("word", "count")
    .limit(20)
)

ham_words = (
    top_words
    .filter(F.col("label") == "ham")
    .select("word", "count")
    .limit(20)
)

display(spam_words)
display(ham_words)


word,count
call,167
you,119
your,93
now,90
the,72
txt,71
free,69
have,67
claim,61
for,59


word,count
lor,140
but,105
then,102
can,94
now,89
got,83
wat,81
not,79
the,64
home,59


In [0]:
spam_top = (
    top_words
    .filter(F.col("label") == "spam")
    .select("word")
    .limit(15)
)

display(spam_top)


word
call
you
your
now
the
txt
free
have
claim
prize


In [0]:
spam_top_clean = (
    top_words
    .filter(F.col("label") == "spam")
    .filter(F.col("word").rlike(".*[a-z].*"))     # musi mieć litery
    .filter(~F.col("word").rlike("^[0-9]+$"))     # nie może być samą liczbą
    .filter(F.length("word") >= 3)               # opcjonalnie: >=3
    .select("word")
    .limit(20)
)

display(spam_top_clean)

spam_list = [r["word"] for r in spam_top_clean.collect()]
print(spam_list)


word
call
you
your
now
the
txt
free
have
claim
prize


['call', 'you', 'your', 'now', 'the', 'txt', 'free', 'have', 'claim', 'prize', 'for', 'from', 'mobile', 'won', 'our', 'are', 'only', 'www', 'this', 'urgent']


In [0]:
interesting_rules = (
    rules
    .filter(F.col("supp_ab") >= F.lit(minsup1))
    .filter(
        F.col("ante").isin(spam_list) |
        F.col("cons").isin(spam_list)
    )
    .orderBy(F.desc("lift"), F.desc("confidence"), F.desc("supp_ab"))
)

display(interesting_rules.select("ante","cons","supp_ab","confidence","lift").limit(50))

ante,cons,supp_ab,confidence,lift
www,ldew,0.005577689243027889,0.16666666666666669,29.880952380952383
ldew,www,0.005577689243027889,1.0,29.88095238095238
12hrs,urgent,0.006374501992031873,0.888888888888889,27.888888888888893
caller,urgent,0.006374501992031873,0.888888888888889,27.888888888888893
urgent,caller,0.006374501992031873,0.2,27.88888888888889
urgent,12hrs,0.006374501992031873,0.2,27.88888888888889
land,urgent,0.006374501992031873,0.8,25.1
urgent,land,0.006374501992031873,0.2,25.1
easy,won,0.008764940239043825,1.0,24.134615384615383
won,easy,0.008764940239043825,0.21153846153846154,24.134615384615383


Prosty filtr spamowy

In [0]:
minsup1  = 0.005   
minconf  = 0.6     
minlift  = 10.0    

In [0]:
df = (
    df
    .withColumn(
        "items",
        F.expr("""
          filter(items, x ->
            x is not null AND (
              x IN ('HAS_URL','HAS_NUMBER','HAS_CURRENCY') OR
              (x rlike '^[a-z]{3,}$')
            )
          )
        """)
    )
    .filter(F.size("items") > 0)
)

train_df, test_df = df.randomSplit([0.7, 0.3], seed=42)
N_train = train_df.count()
N_test  = test_df.count()

print("Train:", N_train)
print("Test :", N_test)

Train: 889
Test : 365


In [0]:
train_items = train_df.withColumn("items_u", F.array_distinct("items"))

In [0]:

N_train = train_df.count()
print("N_train:", N_train)

# 1) Apriori: transakcja jako ZBIÓR (unikalne itemy), posortowane
train_items = (
    train_df
    .withColumn("items_u", F.array_sort(F.array_distinct("items")))
    .filter(F.size("items_u") > 0)
)

# 2) L1: support pojedynczych itemów
L1 = (
    train_items
    .select(F.explode("items_u").alias("item"))
    .groupBy("item")
    .agg(F.count("*").alias("count"))
    .withColumn("supp", F.col("count") / F.lit(N_train))
)

# pomocnicze L1 jako ante/cons
L1a = L1.select(F.col("item").alias("ante"), F.col("supp").alias("supp_ante"))
L1c = L1.select(F.col("item").alias("cons"), F.col("supp").alias("supp_cons"))

# 3) Wszystkie pary (i<j) w obrębie transakcji
pairs = (
    train_items
    .selectExpr("*", "posexplode(items_u) as (i, w1)")
    .withColumn("j", F.explode(F.sequence(F.col("i") + 1, F.size("items_u") - 1)))
    .withColumn("w2", F.expr("get(items_u, j)"))
    .filter(F.col("w2").isNotNull())     
    .select("w1", "w2")
)

# 4) L2: support par
L2 = (
    pairs
    .groupBy("w1","w2")
    .agg(F.count("*").alias("count_ab"))
    .withColumn("supp_ab", F.col("count_ab") / F.lit(N_train))
    .filter(F.col("supp_ab") >= F.lit(minsup1))
)

print("L2 (po minsup) count:", L2.count())
display(L2.orderBy(F.desc("supp_ab")).limit(20))

# 5) Reguły x=>y oraz y=>x + confidence + lift
rules_xy = (
    L2
    .select(F.col("w1").alias("ante"), F.col("w2").alias("cons"), "supp_ab")
    .join(L1a, on="ante", how="left")
    .join(L1c, on="cons", how="left")
    .withColumn("confidence", F.col("supp_ab") / F.col("supp_ante"))
    .withColumn("lift",       F.col("confidence") / F.col("supp_cons"))
)

rules_yx = (
    L2
    .select(F.col("w2").alias("ante"), F.col("w1").alias("cons"), "supp_ab")
    .join(L1a, on="ante", how="left")
    .join(L1c, on="cons", how="left")
    .withColumn("confidence", F.col("supp_ab") / F.col("supp_ante"))
    .withColumn("lift", F.col("confidence") / F.col("supp_cons"))
)

rules_train = (
    rules_xy.unionByName(rules_yx)
    .filter(F.col("supp_ante").isNotNull() & F.col("supp_cons").isNotNull())
    .filter(F.col("confidence") >= F.lit(minconf))
    .filter(F.col("lift") >= F.lit(minlift))
)

print("Liczba reguł (train) po filtrach:", rules_train.count())
display(rules_train.orderBy(F.desc("lift"), F.desc("confidence"), F.desc("supp_ab")).limit(30))


N_train: 889
L2 (po minsup) count: 1203


w1,w2,count_ab,supp_ab
you,you,82,0.0922384701912261
your,your,80,0.08998875140607424
call,you,67,0.07536557930258718
call,your,49,0.05511811023622047
wat,wat,40,0.04499437570303712
call,have,39,0.043869516310461196
call,prize,38,0.04274465691788527
are,you,38,0.04274465691788527
have,you,38,0.04274465691788527
call,now,37,0.04161979752530934


Liczba reguł (train) po filtrach: 267


cons,ante,supp_ab,supp_ante,supp_cons,confidence,lift
within,digital,0.00562429696287964,0.00562429696287964,0.00562429696287964,1.0,177.8
sipix,within,0.00562429696287964,0.00562429696287964,0.00562429696287964,1.0,177.8
yun,yun,0.00562429696287964,0.00562429696287964,0.00562429696287964,1.0,177.8
numbers,ending,0.00562429696287964,0.00562429696287964,0.00562429696287964,1.0,177.8
within,sipix,0.00562429696287964,0.00562429696287964,0.00562429696287964,1.0,177.8
digital,within,0.00562429696287964,0.00562429696287964,0.00562429696287964,1.0,177.8
yun,yun,0.00562429696287964,0.00562429696287964,0.00562429696287964,1.0,177.8
yours,yours,0.00562429696287964,0.00562429696287964,0.00562429696287964,1.0,177.8
ending,numbers,0.00562429696287964,0.00562429696287964,0.00562429696287964,1.0,177.8
sipix,digital,0.00562429696287964,0.00562429696287964,0.00562429696287964,1.0,177.8


In [0]:
# Wybór reguł do filtra spam

# Liczymy jak często reguła odpala w spamach train
spam_train = train_df.filter(F.col("label") == "spam").withColumn("items_u", F.array_distinct("items"))
N_spam_train = spam_train.count()

# Jeżeli N_spam_train=0, to nie dzielimy przez 0
if N_spam_train == 0:
    raise Exception("Brak spamów w train_df — zmień seed lub sprawdź dane.")

# Dla każdej reguły sprawdzamy, w ilu spam SMS-ach występuje (ante i cons)
selected = rules_train.select("ante","cons","supp_ab","confidence","lift")

# Dodatkowy filtr na słowa-klucze
spam_keywords = ["urgent","call","won","free","prize","claim","cash","reply","txt","stop","www","com","HAS_URL","HAS_NUMBER","HAS_CURRENCY"]

selected = selected.filter(F.col("ante").isin(spam_keywords) | F.col("cons").isin(spam_keywords))

print("Reguły po keyword-filter:", selected.count())
display(selected.orderBy(F.desc("lift")).limit(30))

Reguły po keyword-filter: 71


ante,cons,supp_ab,confidence,lift
http,com,0.006749156355455568,1.0,52.294117647058826
representative,cash,0.00562429696287964,1.0,42.333333333333336
tenerife,cash,0.00562429696287964,1.0,42.333333333333336
why,com,0.006749156355455568,0.7499999999999999,39.220588235294116
camcorder,reply,0.010123734533183352,1.0,30.655172413793103
sae,cash,0.007874015748031496,0.7000000000000001,29.633333333333336
caller,urgent,0.006749156355455568,1.0,28.677419354838708
ldew,www,0.00562429696287964,1.0,26.93939393939394
getzed,www,0.00562429696287964,1.0,26.93939393939394
await,cash,0.00562429696287964,0.6249999999999999,26.45833333333333


In [0]:
from functools import reduce

# 4) Klasyfikacja TEST: spam jeśli spełnia ≥ 1 regułę

rules_list = selected.select("ante","cons").collect()

if len(rules_list) == 0:
    raise Exception("Nie znaleziono żadnych reguł po filtrach")

conds = [
    (F.array_contains(F.col("items"), r["ante"]) & F.array_contains(F.col("items"), r["cons"]))
    for r in rules_list
]

rule_trigger = reduce(lambda a,b: a | b, conds)

test_pred = (
    test_df
    .withColumn("predicted", F.when(rule_trigger, F.lit("spam")).otherwise(F.lit("ham")))
)


In [0]:

# 5) Confusion matrix + metryki
cm = (
    test_pred
    .groupBy("label","predicted")
    .count()
    .orderBy("label","predicted")
)

display(cm)

TP = test_pred.filter((F.col("label")=="spam") & (F.col("predicted")=="spam")).count()
FP = test_pred.filter((F.col("label")=="ham")  & (F.col("predicted")=="spam")).count()
FN = test_pred.filter((F.col("label")=="spam") & (F.col("predicted")=="ham")).count()
TN = test_pred.filter((F.col("label")=="ham")  & (F.col("predicted")=="ham")).count()

precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1        = (2*precision*recall/(precision+recall)) if (precision+recall) > 0 else 0.0

print("TP:", TP, "FP:", FP, "FN:", FN, "TN:", TN)
print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)

label,predicted,count
ham,ham,268
ham,spam,1
spam,ham,51
spam,spam,45


TP: 45 FP: 1 FN: 51 TN: 268
Precision: 0.9782608695652174
Recall   : 0.46875
F1       : 0.6338028169014085


In [0]:
# bez selekcji spam keywords

spam_train = train_df.filter(F.col("label") == "spam").withColumn("items_u", F.array_distinct("items"))
N_spam_train = spam_train.count()

if N_spam_train == 0:
    raise Exception("Brak spamów w train_df — zmień seed lub sprawdź dane.")

selected = rules_train.select("ante","cons","supp_ab","confidence","lift")

rules_list = selected.select("ante","cons").collect()

if len(rules_list) == 0:
    raise Exception("Nie znaleziono żadnych reguł po filtrach")

conds = [
    (F.array_contains(F.col("items"), r["ante"]) & F.array_contains(F.col("items"), r["cons"]))
    for r in rules_list
]

rule_trigger = reduce(lambda a,b: a | b, conds)

test_pred = (
    test_df
    .withColumn("predicted", F.when(rule_trigger, F.lit("spam")).otherwise(F.lit("ham")))
)

cm = (
    test_pred
    .groupBy("label","predicted")
    .count()
    .orderBy("label","predicted")
)

display(cm)

TP = test_pred.filter((F.col("label")=="spam") & (F.col("predicted")=="spam")).count()
FP = test_pred.filter((F.col("label")=="ham")  & (F.col("predicted")=="spam")).count()
FN = test_pred.filter((F.col("label")=="spam") & (F.col("predicted")=="ham")).count()
TN = test_pred.filter((F.col("label")=="ham")  & (F.col("predicted")=="ham")).count()

precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1        = (2*precision*recall/(precision+recall)) if (precision+recall) > 0 else 0.0

print("TP:", TP, "FP:", FP, "FN:", FN, "TN:", TN)
print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)


label,predicted,count
ham,ham,181
ham,spam,88
spam,ham,22
spam,spam,74


TP: 74 FP: 88 FN: 22 TN: 181
Precision: 0.4567901234567901
Recall   : 0.7708333333333334
F1       : 0.5736434108527132
